# DeepHarvest — Data

| Source | Content | Coverage |
|---|---|---|
| USDA NASS Quick Stats | County yield + acres harvested, corn & soybeans | 2,695 counties, 1980–2025 |
| NASA POWER (`community=AG`) | Daily tmax, tmin, precip, solar radiation | same counties, 1990–today |
| Census Gazetteer | County centroids (lat/lon, land area) | — |

Reference ET (`et0`) is computed locally with Hargreaves, not fetched.

In [3]:
import sys
from pathlib import Path

# Walk up to the repo root so this works from notebooks/ or the project root.
ROOT = Path.cwd()
while not (ROOT / "pipeline").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (str(ROOT / "pipeline"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd

from dataset import (STATIC_COLS, acre_weighted, holdout_tail, load_features,
                     metrics, usable_mask, weather_cols, year_split)

CROP      = "corn"     # "corn" | "soybeans"
AS_OF_BIN = 14         # 14 = full season; 10 = what 2026 has observed so far

df, W, S = load_features(CROP, AS_OF_BIN)

y      = df["yield"].to_numpy(dtype=float)
usable = usable_mask(df)                  # labeled AND has a prior year
X      = df[weather_cols(AS_OF_BIN) + STATIC_COLS]   # flat design matrix

print(f"df {df.shape}   W {W.shape}   S {S.shape}   X {X.shape}")
print(f"{df.fips.nunique():,} counties | {df.year.min()}–{df.year.max()}")
print(f"labeled {int(df['yield'].notna().sum()):,} | usable {int(usable.sum()):,}")
df.head(3)
df.to_csv("data.csv", index=False)

df (99715, 144)   W (99715, 14, 8)   S (99715, 12)   X (99715, 124)
2,695 counties | 1990–2026
labeled 64,057 | usable 57,147


## What's in `df`

**99,715 rows × 144 columns**, one row per **county-year**, sorted by `(fips, year)`.
Source: `data/processed/features_{crop}.parquet`.

| Group | Cols | What |
|---|---|---|
| `fips`, `year` | 2 | Key. `fips` is a zero-padded **5-char string** — keep it a string |
| `yield`, `acres` | 2 | Target (bu/acre) + acres harvested |
| `state_alpha`, `county_name`, `crop` | 3 | Labels |
| `lat`, `lon`, `land_sqmi` | 3 | Geography |
| `yield_lag1/2/3`, `yield_prior_mean`, `yield_prior_std`, `n_prior_years`, `trend_slope`, `trend_pred` | 8 | Yield history |
| `{gdd, precip, tmax_mean, tmax_max, heat_days, et0, radiation, water_balance}_t0..t13` | 112 | Season weather: 8 channels × 14 biweekly bins |
| `n_days_t0..t13` | 14 | Days observed per bin (`0` = unobserved) |

`W` is the 112 weather columns reshaped to `[N, 14, 8]` for sequence models; `S` is the
8 history columns + `lat/lon/land_sqmi/year` as `[N, 12]`. **Same row order as `df`**, so
`df.iloc[i]`, `W[i]` and `S[i]` are the same county-year.

Bins span Apr 1 (DOY 91) → Oct 31 (DOY 304), ~15.3 days each.

### Nulls — both intentional

- **35,658 rows have `yield` null.** Weather covers every county-year 1990–2026, but NASS
  only reports where the crop is actually grown. Use `usable_mask(df)` → 57,147 rows
  (also requires a prior year, for the lag features).
- **`acres` has ~500 more nulls than `yield`.** NASS sometimes publishes a yield but
  suppresses acreage. `acre_weighted()` falls back to an unweighted mean rather than
  dropping the county.

In [2]:
# install packages into kernel
!pip install numpy pandas scikit-learn tensorflow

  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 MB 48.6 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 45.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 38.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.4/565.4 kB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 45.5 MB/s  0:00:00 eta 0:00:01
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [tensorflow]1 [tensorflow]
